In [1]:
%load_ext autoreload
%autoreload 2
import os
import shutil
import hydra
from scipy.spatial.transform import Rotation as R
import pathlib
from pathlib import Path
import wandb
import torch
import tqdm
import dill
import json
import numpy as np
import datetime
import matplotlib.pyplot as plt
from diffusion_policy.workspace.base_workspace import BaseWorkspace
from diffusion_policy.env_runner.robomimic_image_runner_joint_space import RobomimicImageRunnerJointSpace
from diffusion_policy.policy.diffusion_unet_hybrid_image_policy import DiffusionUnetHybridImagePolicy

from hydra import compose, initialize
from omegaconf import OmegaConf

config_path = Path("diffusion_policy/config/train_diffusion_unet_hybrid_workspace.yaml")
DATASET_PATH = "/data/scene-rep/u/iyu/scene-jacobian-discovery/diff-policy/diffusion_policy/data/robomimic/datasets/lift/ph/image_abs.hdf5"
overrides = [
    # "config-name=train_robomimic_image_joint_space_workspace.yaml",
    "task=lift_image_abs_joint_space",
    f"task.env_runner.dataset_path={DATASET_PATH}",
    "task.env_runner.n_train=0",
    "task.env_runner.n_train_vis=0",
    "task.env_runner.n_test=1",
    "task.env_runner.n_test_vis=1",
]

OmegaConf.register_new_resolver("eval", eval, replace=True)
with initialize(version_base=None, config_path=str("../../" / config_path.parent)):
    cfg = compose(config_name=str(config_path.name), overrides=overrides)

OmegaConf.resolve(cfg)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["MUJOCO_GL"] = "egl"
os.environ["DISPLAY"] = ":1"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env_runner = hydra.utils.instantiate(cfg.task.env_runner, output_dir=None)

/home/iyu/miniconda3/envs/robodiff-orig/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


env kwargs {'has_renderer': False, 'has_offscreen_renderer': True, 'ignore_done': True, 'use_object_obs': False, 'use_camera_obs': True, 'control_freq': 20, 'controller_configs': {'type': 'JOINT_POSITION', 'input_max': 1, 'input_min': -1, 'output_max': 0.05, 'output_min': -0.05, 'kp': 50, 'damping_ratio': 1, 'impedance_mode': 'fixed', 'kp_limits': [0, 300], 'damping_ratio_limits': [0, 10], 'qpos_limits': None, 'interpolation': None, 'ramp_ratio': 0.2}, 'robots': ['Panda'], 'camera_depths': False, 'camera_heights': 84, 'camera_widths': 84, 'reward_shaping': False, 'camera_names': ['agentview', 'robot0_eye_in_hand'], 'render_gpu_device_id': 0}


### Load the dataset

In [33]:
import h5py
import mediapy as media
f = h5py.File(DATASET_PATH, "r")
demos = sorted(list(f["data"].keys()))
# extract filter key information
if "mask" in f:
    all_filter_keys = {}
    for fk in f["mask"]:
        fk_demos = sorted([elem.decode("utf-8") for elem in np.array(f["mask/{}".format(fk)])])
        all_filter_keys[fk] = fk_demos

# put demonstration list in increasing episode order
inds = np.argsort([int(elem[5:]) for elem in demos])
demos = [demos[i] for i in inds]
# extract length of each trajectory in the file
traj_lengths = []
action_min = np.inf
action_max = -np.inf
joint_pos_min = np.inf
joint_pos_max = -np.inf
for ep in demos:
    traj_lengths.append(f["data/{}/actions".format(ep)].shape[0])
    action_min = min(action_min, np.min(f["data/{}/actions".format(ep)][()]))
    action_max = max(action_max, np.max(f["data/{}/actions".format(ep)][()]))
    joint_pos_min = min(joint_pos_min, np.min(f["data/{}/obs/robot0_joint_pos".format(ep)][()]))
    joint_pos_max = max(joint_pos_max, np.max(f["data/{}/obs/robot0_joint_pos".format(ep)][()]))
traj_lengths = np.array(traj_lengths)

# report statistics on the data
print("")
print("total transitions: {}".format(np.sum(traj_lengths)))
print("total trajectories: {}".format(traj_lengths.shape[0]))
print("traj length mean: {}".format(np.mean(traj_lengths)))
print("traj length std: {}".format(np.std(traj_lengths)))
print("traj length min: {}".format(np.min(traj_lengths)))
print("traj length max: {}".format(np.max(traj_lengths)))
print("action min: {}".format(action_min))
print("action max: {}".format(action_max))
print("")

ep = demos[10]
robot_joint_pos = f["data/{}/{}/{}".format(ep, "next_obs", "robot0_joint_pos")] 
gripper_qpos = f["data/{}/{}".format(ep, "actions")][:, -1:]  # gripper is the last action dim
# get the images
print("robot_joint_pos shape: ", robot_joint_pos.shape)
print("gripper_qpos shape: ", gripper_qpos.shape)
playback_actions = np.concatenate([robot_joint_pos, gripper_qpos], axis=-1)
print("playback_actions shape: ", playback_actions.shape)
print("playback_actions: ", playback_actions)


total transitions: 9666
total trajectories: 200
traj length mean: 48.33
traj length std: 6.116461395284041
traj length min: 36
traj length max: 64
action min: -2.764532568641088
action max: 2.7911990332617242

robot_joint_pos shape:  (54, 7)
gripper_qpos shape:  (54, 1)
playback_actions shape:  (54, 8)
playback_actions:  [[ 1.44126088e-02  1.94978823e-01 -8.31212117e-03 -2.65112180e+00
   2.21056321e-05  2.94716046e+00  7.93057381e-01 -1.00000000e+00]
 [ 1.44680791e-02  1.97579004e-01 -6.63349865e-03 -2.64486392e+00
  -1.85955837e-05  2.93928503e+00  7.94216722e-01 -1.00000000e+00]
 [ 1.52443697e-02  2.03227023e-01 -2.93555882e-03 -2.63368806e+00
  -4.50109955e-04  2.92677820e+00  7.99687527e-01 -1.00000000e+00]
 [ 1.64934648e-02  2.15943559e-01  6.94299819e-04 -2.61608116e+00
  -2.57453034e-03  2.91239557e+00  8.09896594e-01 -1.00000000e+00]
 [ 1.83551690e-02  2.38588772e-01  4.05322981e-03 -2.59091509e+00
  -7.10540035e-03  2.89995228e+00  8.24976065e-01 -1.00000000e+00]
 [ 2.084920

In [31]:
import mediapy as media
from diffusion_policy.env.robomimic.lift_joint_space import LiftJointSpace
from diffusion_policy.env_runner.robomimic_image_runner_joint_space import create_env
from diffusion_policy.env.robomimic.robomimic_image_wrapper import RobomimicImageWrapper
import robomimic.utils.file_utils as FileUtils


def playback(env: RobomimicImageWrapper, actions: np.array):
    vid = []
    pbar = tqdm.tqdm(total=actions.shape[0])
    print("playback actions shape: ", actions.shape)

    for i in range(0, actions.shape[0]):
        obs, reward, done, info = env.step(actions[i])
        pbar.update()
        vid.append(env.render())
    pbar.close()
    return vid

env_meta = FileUtils.get_env_metadata_from_dataset(
    DATASET_PATH)
# disable object state observation
env_meta['env_kwargs']['use_object_obs'] = False
env_meta['env_kwargs']['controller_configs'] = {
    "type": "JOINT_POSITION",
    "input_max": 1,
    "input_min": -1,
    "output_max": [0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.5],
    "output_min": [-0.05, -0.05, -0.05, -0.05, -0.05, -0.05, -0.05, -0.5],
    "kp": 150,
    "damping_ratio": 10,
    "impedance_mode": "fixed",
    "kp_limits": [0, 300],
    "damping_ratio_limits": [0, 10],
    "qpos_limits": None,
    "interpolation": None,
    "ramp_ratio": 0.2,
    "input_type": "absolute"
  }
robomimic_env = create_env(
                env_meta=env_meta, 
                env_target=cfg.task.env_runner.env_target,
                shape_meta=cfg.task.shape_meta,
            )
robomimic_env.env.hard_reset = False
import robomimic.utils.obs_utils as obs_utils
obs_utils.initialize_obs_utils_with_obs_specs({"obs": {"low_dim": ["robot0_joint_pos", "robot_0_gripper_qpos"], "rgb": ["agentview_image", "robot0_eye_in_hand_image"]}})
robomimic_image_runner = RobomimicImageWrapper(
                env=robomimic_env,
                shape_meta=cfg.task.shape_meta,
                init_state=None,
                render_obs_key='agentview_image'
            )


============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot_0_gripper_qpos', 'robot0_joint_pos']
using obs modality: rgb with keys: ['agentview_image', 'robot0_eye_in_hand_image']


In [54]:
robomimic_image_runner.reset()
playback_actions_test = playback_actions.copy()
playback_actions_test[:, [0, 1, 2, 4, 5, 6, 7]] = 0.0
# get the joint min max for each joint
joint_min = np.min(playback_actions[1:] - playback_actions[:-1], axis=0)
joint_max = np.max(playback_actions[1:] - playback_actions[:-1], axis=0)
print("joint min: ", joint_min)
print("joint max: ", joint_max)

video = playback(robomimic_image_runner, playback_actions_test)
media.show_video(video, fps=30)

joint min:  [-0.00129625 -0.04413067 -0.00357685 -0.04206041 -0.01318722 -0.02199442
 -0.01057208  0.        ]
joint max:  [0.00460912 0.03572855 0.00381664 0.03598165 0.00564048 0.01371863
 0.02898276 2.        ]


playback actions shape:  (54, 8)























































100%|██████████| 54/54 [00:08<00:00,  6.03it/s]
